# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — Exploration with `mlcroissant`

This notebook guides you through loading, exploring, and analyzing the FAIR² dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library.

### Dataset Source
The dataset is described using the [MLCommons Croissant schema](https://mlcommons.org/croissant/), accessible via the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Examine the available record sets, fields, and their `@id` identifiers in the dataset.

First, list all available record sets by their `@id`, name, and a sample of their fields.

In [ ]:
# List all record sets and their properties
if hasattr(metadata, 'record_sets') and len(metadata.record_sets) > 0:
    print('Available record sets:')
    for rs in metadata.record_sets:
        print(f"\n@id: {rs.id}")
        print(f"  name: {getattr(rs, 'name', 'N/A')}")
        if hasattr(rs, 'fields') and rs.fields:
            print("  Fields:")
            for field in rs.fields:
                print(f"    - @id: {field.id} (name: {getattr(field, 'name', 'N/A')})")
else:
    print('No record sets found in dataset metadata.')

For demonstration (due to the dynamic schema), we'll display a preview of data from the first available record set, referencing it by its `@id`. For production analysis, always reference by `@id`.

In [ ]:
# Get the list of all available record set `@id`s
record_set_ids = []
if hasattr(metadata, 'record_sets') and len(metadata.record_sets) > 0:
    for rs in metadata.record_sets:
        record_set_ids.append(rs.id)
else:
    print('No record sets found!')
    record_set_ids = []

# If there are record sets, preview the first one
first_rs_id = record_set_ids[0] if record_set_ids else None
if first_rs_id:
    print(f"Previewing records from record set @id: {first_rs_id}\n")
    for i, x in enumerate(dataset.records(record_set=first_rs_id)):
        print(x)
        if i >= 2:
            break
else:
    print('No records to display.')

## 3. Data Extraction

Load data from each record set into a `pandas.DataFrame` by referencing each record set's `@id`. This enables tabular analysis and further processing.

In [ ]:
# Extract all available record sets into pandas DataFrames, indexed by their @id
dataframes = {}

for record_set_id in record_set_ids:
    rows = list(dataset.records(record_set=record_set_id))
    if rows:
        dataframes[record_set_id] = pd.DataFrame(rows)
        print(f"Loaded DataFrame for record set @id: {record_set_id}. Shape: {dataframes[record_set_id].shape}")
    else:
        print(f"Record set {record_set_id} did not yield any data.")

# Display columns for the first extracted DataFrame
if first_rs_id and first_rs_id in dataframes:
    print("\nColumns in DataFrame:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()
else:
    print('No DataFrames loaded.')

## 4. Exploratory Data Analysis (EDA)

Demonstrate basic filtering, normalization, and grouping using only entity `@id`s.

Here, we:
- Filter records where a numeric field exceeds a threshold
- Normalize this numeric field
- Group by a categorical field if present

All entity references are by their `@id`.

**Adapt the field `@id` calls to your schema as required.**

In [ ]:
# Set your record set and field `@id`s for this analysis
rs_analysis = first_rs_id  # Use the first available record set @id

# Try to pick the first numeric field for demonstration
numeric_field_id = None
group_field_id = None

if rs_analysis:
    # Get the record set metadata object
    rs_metadatas = [rs for rs in metadata.record_sets if rs.id == rs_analysis]
    rs_meta = rs_metadatas[0] if rs_metadatas else None
    if rs_meta and hasattr(rs_meta, 'fields'):
        # Try to locate a numeric field and a group/categorical field
        for f in rs_meta.fields:
            if hasattr(f, 'data_type') and (f.data_type in ['schema:Integer', 'schema:Number', 'schema:Float']):
                numeric_field_id = f.id
                break
        for f in rs_meta.fields:
            if hasattr(f, 'data_type') and (f.data_type in ['schema:Text','schema:DefinedTerm']):
                group_field_id = f.id
                break

# Proceed if we have the necessary fields
if rs_analysis and rs_analysis in dataframes and numeric_field_id in dataframes[rs_analysis].columns:
    df = dataframes[rs_analysis]
    # Ensure numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean()  # Just for demo, use mean as threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by group_field_id if possible
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
    else:
        print("\nNo categorical field found to group by.")
else:
    print("Insufficient data or field references for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field (referenced by its `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if rs_analysis and rs_analysis in dataframes and numeric_field_id in dataframes[rs_analysis].columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[rs_analysis][numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion

- This notebook demonstrated how to load, preview, and analyze a Croissant-formatted dataset using the `mlcroissant` library.
- All references to dataset entities (record sets, fields) were done strictly using their `@id` fields.
- Further exploration can include modeling, deeper statistical analysis, and custom visualization—all beginning with rigorous, schema-aware data loading.

For more information and advanced usage, see the [mlcroissant documentation](https://mlcroissant.readthedocs.io/).